In [0]:
import os, sys, json
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import fhir_demo as fd

cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"))
state_file = os.path.join(os.path.dirname(cl.raw_root(cfg)), "_demo", "scd2_demo_patients.json")
state = json.load(open(state_file))
if "updated" in state:
    raise Exception(f"Already updated: {state['updated']}")

def alpha_change(res):                       # tracked columns change
    res["address"][0]["city"] = "Mumbai"
    res["maritalStatus"]["coding"][0].update({"code": "M", "display": "Married"})

def bravo_change(res):                       # only an untracked field changes
    res.setdefault("identifier", []).append(
        {"system": "urn:scd2-demo", "value": state["run_tag"] + "-touch"})

state["updated"] = {
    "alpha": fd.update_test_patient(cfg, state["alpha"]["id"], alpha_change),
    "bravo": fd.update_test_patient(cfg, state["bravo"]["id"], bravo_change),
}
print(state["updated"])                       # both should show version 2
with open(state_file, "w") as f:
    json.dump(state, f)
print("Load date to use:", state["load_date"])